# Signal quality report

Load a recording by path, compute quality metrics over a shared grid of intervals,
apply filters to the joined table, then visualize whatever got flagged.

The pipeline is four independent stages, which is the point of the library:

| stage | what it does | what it does *not* do |
|---|---|---|
| `load` | path → `Recording` (xarray) | interpret anything |
| `compute` | metrics → one joined table | decide what is bad |
| `apply_filters` | table → flags | recompute anything |
| `viz` | flags → evidence | hide what it excluded |

Because filtering is separate from measurement, you can re-judge the same table
under a different policy without recomputing a thing.

> **Data stays outside this repo.** Point `STUDY` at a recording on disk. Before
> sharing this notebook, clear outputs or de-identify them — channel names and
> annotations can carry identifying information.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import signal_quality as sq
from signal_quality import metrics as M

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.width", 200)

# An XLTEK study directory, a lossless .h5 archive, or any file MNE can read.
STUDY = Path("~/Documents/example_study").expanduser()
LINE_FREQ = 60.0  # 60 in the Americas, 50 across most of the rest of the world

## 1. Load

`load()` sniffs the path. XLTEK studies and lossless HDF5 archives keep their raw
ADC counts and acquisition stamp tables, which is what makes clipping and
timestamp checks possible; other formats load fine but those checks report
themselves unavailable rather than guessing.

In [ ]:
rec = sq.load(STUDY, line_freq=LINE_FREQ)
print(rec)
print("raw counts available:", rec.has_counts)
print("stamp tables available:", "stamps" in rec.provenance)
rec.ds

## 2. Generic integrity checks

These are recording-scope and signal-agnostic — they ask whether the data exists,
whether its clock is coherent, and whether the channels can honestly share one
time axis. They are kept in their own table rather than broadcast across every
channel row, because a gap is a property of the recording, not of any channel.

In [ ]:
issues = sq.check_integrity(rec)
issues

In [ ]:
sq.viz.plot_availability(rec, issues)
plt.show()

## 3. Compute metrics over a shared interval grid

Every metric is computed over the *same* `IntervalGrid`, so the results share a
row index exactly and the join is a plain concat — no alignment guesswork, no
NaN padding.

`IntervalGrid.whole` gives one row per channel (a whole-recording summary);
`IntervalGrid.fixed(rec, 30.0)` gives a per-window trend. Section 6 uses the
windowed form for the same metrics.

In [ ]:
grid = sq.IntervalGrid.whole(rec)

mf = sq.compute(
    [
        M.RMS(),              # broadband amplitude
        M.LineRatio(),        # mains pickup -> contact quality
        M.EMGFraction(),      # muscle contamination
        M.MaxCorrelation(),   # isolation
        M.FlatFraction(),     # dead / flat stretches
        M.ClipFraction(),     # converter saturation (needs raw counts)
    ],
    rec,
    grid,
    ch_type="eeg",
)

for note in mf.notes:  # metrics that could not be computed, and why
    print("note:", note)

mf.table.droplevel("interval").drop(columns=["t_start", "t_end", "coverage"]).sort_values(
    "line_ratio", ascending=False
).round(2)

## 4. Apply filters

Thresholds live here, separate from the measurements, so re-judging costs
nothing. `DEFAULT_FILTERS` carries the defaults tuned for clinical scalp EEG —
treat them as a starting point for any other modality.

Each flag records the value and threshold that produced it, so no verdict is
unfalsifiable.

In [ ]:
flags = sq.apply_filters(mf, sq.DEFAULT_FILTERS)
verdicts = sq.verdict(flags, channels=mf.table.index.get_level_values("channel").unique())

print(verdicts["verdict"].value_counts().to_dict())
verdicts[verdicts["verdict"] != "good"]

In [ ]:
# Which issues are present, and how widespread each one is.
flags.groupby(["flag", "severity"])["channel"].agg(["nunique", lambda s: ", ".join(sorted(set(s)))]).rename(
    columns={"nunique": "n_channels", "<lambda_0>": "channels"}
)

## 5. Visualize what was flagged

### 5.1 Electrode positions on the head model

Left: the metric interpolated across the scalp — illustrative, since only the
electrode sites carry measurements. Right: the authoritative per-electrode
verdict, unsmoothed, labelled with the flags that fired.

In [ ]:
sq.viz.plot_contact_quality(mf, verdicts, metric="line_ratio", log=True)
plt.show()

### 5.2 Spectra: flagged vs unflagged channels

A table saying a channel has a mains ratio of 900 is an assertion; a spectrum
showing its line-frequency spike towering over every clean channel is the
evidence. Check every flag class this way before acting on it.

In [ ]:
if "LINE_NOISE" in set(flags["flag"]):
    sq.viz.plot_good_bad_psd(rec, flags, flag="LINE_NOISE")
    plt.show()
else:
    print("no line-noise flags; showing the spread instead")
    sq.viz.plot_psd_examples(rec, mf, metric="line_ratio")
    plt.show()

### 5.3 The raw signal behind the flags

Worst flagged channels above, best unflagged below, same window and same gain.
This is where a "flat" channel that actually has a visible trace, or a clipping
flag driven by three stray samples, gives itself away.

In [ ]:
present = [f for f in ["LINE_NOISE", "AMP_OUTLIER", "FLAT", "CLIPPING"] if f in set(flags["flag"])]
metric_for = {
    "LINE_NOISE": "line_ratio",
    "AMP_OUTLIER": "rms",
    "FLAT": "flat_frac",
    "CLIPPING": "clip_pct",
}
for flag in present[:2]:
    sq.viz.plot_flagged_examples(rec, mf, flags, flag=flag, metric=metric_for[flag], n=3)
    plt.show()

### 5.4 Highly correlated pairs

A shortlist, not a verdict. Near-unity correlation is consistent with a salt
bridge, but on a common-reference recording the shared reference and drift push
almost every pair toward 1 — which is exactly why this library ships no
automatic bridge filter. Confirm a candidate against a bipolar derivation: a
bridged pair collapses to a much smaller amplitude than a healthy one.

In [ ]:
M.correlation_pairs(rec, threshold=0.95, top=10)

## 6. Quality over time

The same metrics, recomputed on a windowed grid. Nothing about the metrics or
filters changes — only the grid does. A recording can be perfectly continuous
and still be unusable for stretches, which is what the artifact trace shows.

In [ ]:
wgrid = sq.IntervalGrid.fixed(rec, 30.0)
wmf = sq.compute([M.RMS(), M.LineRatio(), M.PeakToPeak()], rec, wgrid, ch_type="eeg")

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
sq.viz.plot_metric_trend(wmf, "line_ratio", rec=rec, ax=axes[0], threshold=300)
sq.viz.plot_clean_fraction(wmf, rec=rec, metric="p2p", threshold=150.0, ax=axes[1])
fig.tight_layout()
plt.show()

## 7. Summary

Everything below is derived from the signal itself. Use it to decide which
channels to exclude from quantitative analysis — and note that excluding them
is a judgement call the library deliberately leaves to you.

In [ ]:
bad = verdicts[verdicts["verdict"] == "bad"].index.tolist()
marginal = verdicts[verdicts["verdict"] == "marginal"].index.tolist()

print(f"recording      : {rec}")
print(f"missing data   : {100 * (1 - rec.covered.mean()):.1f}% of the timeline")
print(f"integrity      : {len(issues)} finding(s) — {sorted(set(issues['check'])) if len(issues) else 'none'}")
print(f"bad channels   : {len(bad)} — {', '.join(bad) if bad else 'none'}")
print(f"marginal       : {len(marginal)} — {', '.join(marginal) if marginal else 'none'}")
print()
print("Suggested exclusion list for quantitative analysis:")
print("  BAD =", bad)